# Task 2. Clean the prices and build the weekly tables

This notebook turns the original daily price file into three tables for Task 3:

- **train:** weeks from 2013 to 2015
- **validation:** weeks in 2016
- **test:** weeks in 2017

Each row is one stock for one week. The value to predict is next week's volatility. The features use only information known on that week's anchor morning.

The original file is the brief link `https://raw.githubusercontent.com/plotly/datasets/master/all_stocks_5yr.csv`. Its check code is `6aea253cd19de60b568143991aaf1fa482456565c389205658d236e595e716cf`.

Wikinews is not in this notebook. The price tables can go to Task 3 without it.

## 1. Rules

The original file is never edited. Rejected rows are written to their own table with a reason.

- A row after 29 December 2017, or before 8 February 2013, is `out_of_range`.
- A row inside that window is broken if a price is blank or not a number, if high is below low, if open or close is outside the high-low range, or if volume is blank, negative, or zero.
- Stock-split weeks stay in the data. A week whose price jumps by 30% or more is marked with `split_flag`. It is not deleted. The one broken LNT row on 19 May 2016 is rejected because its close is outside the day's high-low range.
- APTV, BHF, BHGE, DWDP, and DXC have less than one year of prices inside the window, so they stay in the cleaned daily file and are left out of the model tables.

The anchor is the last trading day in a Monday-Friday week. If Friday is a holiday, the anchor is the earlier trading day in that week. Prediction time is that morning: the open is known, and that day's high, low, close, and volume are not used as features.

Features are past volatility over 1, 4, 12, and 52 weeks, the completed week's high-low range, the completed week's return, and that week's volume divided by the stock's own median volume over the previous 12 weeks. The 52-week volatility is missing for early weeks. Those blanks are filled with the **training** median only, then the same number is used for 2016 and 2017.

A row keeps its target only when the next anchor is still inside the same split. A December 2015 week whose next week falls in 2016 is left out of training. It is not moved into 2016.

## 2. Laptop check

The cell below reads the check that was already run with `python task2/run_local.py`. That run uses the same rules as the Spark job. It is a development check. It does not replace the EMR run.

Rebuild it, from the project folder, with:

`python task2/run_local.py`

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path('task2_outputs/reconciliation.json').read_text(encoding='utf-8'))
print('file check', summary['sha256'])
print('total', summary['total_rows'])
print('rejected', summary['rejected_rows'])
print('accepted', summary['accepted_rows'])
print('reasons', summary['reason_counts'])
print('weekly rows', summary['split_counts'])
print('52-week training median', summary['vol_52w_training_median'])
print('rows that add up', summary['identity_ok'])

## 3. What the check found

The file has **619,040** daily rows and 505 stocks. Dates run from 8 February 2013 to 7 February 2018.

- **13,130** rows are after 29 December 2017.
- **23** rows inside the window are broken. Some of those 23 have more than one problem, so the reason counts add up to more than 23.
- **605,887** rows are kept. This is the accepted-bar count in the brief.
- 619,040 = 13,130 + 23 + 605,887.

The weekly model tables have **66,907** training rows (10 May 2013 to 24 December 2015), **25,310** validation rows (8 January 2016 to 23 December 2016), and **25,441** test rows (6 January 2017 to 22 December 2017). No target week crosses into the next split.

On 116,166 pairs of consecutive weeks, the target on one row is exactly the 1-week volatility on the next row. That is the check that the target is next week's volatility and is not also being used as a feature of the current week.

Thirty weeks are flagged for a price jump of 30% or more, including the DISCA, EBAY, and NI split weeks. Those weeks stay in the tables. The broken LNT close on 19 May 2016 is in `task2_outputs/rejected_rows.csv` and is not in the weekly tables.

The first training weeks do not have 52 weeks of history. 19,103 training rows, 268 validation rows, and 78 test rows get the training median in `vol_52w`. The median is about 0.01317.

## 4. Cluster run

The graded run uses four tools. From the project folder on the EMR notebook machine:

1. **HDFS.** `task2/run_emr.sh` puts the original CSV in `/project/raw/prices/` and does not change the file.
2. **Hadoop Streaming.** `task2/streaming/mapper.py` and `reducer.py` count accepted rows, rejected rows, and reasons.
3. **PySpark.** `task2/spark/prepare_features.py` writes the rejected rows, the cleaned daily rows, and the three weekly tables to `/project/prepared/`.
4. **Hive.** `task2/hive/01_register.sql` registers the tables. `task2/hive/02_reconcile.sql` checks the counts. Hive on Tez is fine. Spark SQL does not replace this step.

Run `bash task2/run_emr.sh` and save the printed job output. This laptop does not have Spark or Hive, so that script has not been run here. The cluster settings are still provisional: EMR 7.10.0, one primary and two core m5.xlarge nodes, 100 GiB per core node, replication 2.

## 5. Hand-off to Task 3

Task 3 should train on `weekly_train`, choose the model with `weekly_validation`, and score `weekly_test` once.

Use these columns as features: `vol_1w`, `vol_4w`, `vol_12w`, `vol_52w`, `high_low_range`, `weekly_return`, `volume_vs_normal`, and `split_flag`.

The target column is `target_next_vol`. Do not use `ticker`, `anchor_date`, or `next_anchor_date` as features. Do not fit a new fill value for `vol_52w`; that fill is already taken from the training rows only.

Do not change the model after looking at the 2017 scores.